# 🖊️ PROYECTO · PAPELERÍA CON VOZ — el punto de venta de "Papelería La Escuadra" ✂️

Construyes un **punto de venta con voz** para una papelería. El cliente dicta su pedido, el sistema entiende **artículo y cantidad**, arma el **carrito**, calcula el **total de la venta** y, al **cerrar la venta**, lo guarda en un **historial (SQLite)** para generar un **dashboard**.

| Módulo | Qué hace |
|---|---|
| 🔤 **Catálogo fijo** | Artículos de papelería con su precio |
| 🎙️ **Voz → texto** | El cliente dicta su pedido (artículo + cantidad) |
| 🧠 **Parser** | Extrae artículo y cantidad (regex + alias/sinónimos) |
| 🧾 **Carrito** | Acumula artículos y muestra subtotales |
| 💰 **Total** | Calcula el total de la venta |
| 💾 **Historial** | Guarda cada venta en **SQLite** (fecha, artículos, total) |
| 📊 **Dashboard** | Gráficas de ventas por día, por artículo y resumen |
| 🖥️ **App Gradio** | Interfaz con micrófono y botón "Cerrar venta" |

> ✅ Corre en **CPU**, sin API keys externas (la voz usa Google Speech vía `speech_recognition`). Todo es local.

Corre cada celda con **▶** en orden. 🚀

## 🔧 Paso 1 · Instalar las herramientas
Instalamos librerías para voz (`speech_recognition`), la app (`gradio`) y datos/gráficas (`pandas`, `plotly`). SQLite viene incluido en Python.

In [ ]:
%%capture
!pip install -q speech_recognition gradio pandas plotly numpy gtts
!apt-get -qq install -y portaudio19-dev libportaudio2 flac ffmpeg > /dev/null 2>&1
print("✔ Dependencias listas")

## 📋 Paso 2 · El catálogo de la papelería (la única verdad de precios)
Aquí vive **qué se vende y a cuánto**. Todo el sistema usa este diccionario como fuente única: el parser y el total nunca inventan un precio.

In [ ]:
# Catálogo fijo de Papelería La Escuadra.
# Cada artículo tiene "alias": sinónimos con los que un cliente lo dice por voz.
CATALOGO = {
    "cuaderno profesional rayado": {"precio": 42.00, "alias": ["cuaderno profesional", "cuaderno rayado", "cuaderno"]},
    "cuaderno de cuadro chico":      {"precio": 38.00, "alias": ["cuaderno de cuadro", "cuaderno cuadriculado", "cuaderno cuadro"]},
    "pluma azul":                    {"precio": 8.00,  "alias": ["pluma", "boligrafo azul", "boli azul", "plumas"]},
    "pluma negra":                   {"precio": 8.00,  "alias": ["pluma negra", "boligrafo negro", "boli negro"]},
    "pluma roja":                    {"precio": 9.00,  "alias": ["pluma roja", "boligrafo rojo", "boli rojo"]},
    "lapiz":                         {"precio": 6.00,  "alias": ["lapices", "lapices de madera"]},
    "marcatextos amarillo":          {"precio": 12.00, "alias": ["marcatextos", "marca textos", "resaltador", "marcatexto"]},
    "juego de geometria":            {"precio": 55.00, "alias": ["juego de geometria", "juego geometrico", "geometria", "escuadras"]},
    "calculadora cientifica":        {"precio": 210.00,"alias": ["calculadora", "calculadora cientifica"]},
    "hojas de colores":              {"precio": 25.00, "alias": ["hojas de color", "papel de colores"]},
    "papel bond tamano carta":       {"precio": 1.50,  "alias": ["hojas blancas", "hojas", "papel bond", "hojas sueltas"]},
    "carpeta":                       {"precio": 18.00, "alias": ["carpetas", "folder", "folders"]},
    "sacapuntas":                    {"precio": 10.00, "alias": ["tajador", "sacapuntes"]},
    "goma de borrar":                {"precio": 7.00,  "alias": ["goma", "borrador", "goma blanca"]},
    "tijeras":                       {"precio": 22.00, "alias": ["tijera"]},
    "resistol 5000":                 {"precio": 15.00, "alias": ["resistol", "pegamento", "pegamento blanco", "cola blanca"]},
    "cinta adhesiva":                {"precio": 14.00, "alias": ["cinta", "cinta transparente", "diurex", "cinta scotch", "sinta"]},
    "cinta canela":                  {"precio": 13.00, "alias": ["cinta canela", "tape", "cinta masking", "cinta de papel"]},
    "corrector de cinta":            {"precio": 32.00, "alias": ["corrector", "liquid paper", "correcto", "corrector de cinta"]},
    "regla 30 cm":                   {"precio": 11.00, "alias": ["regla"]},
    "colores normales":              {"precio": 45.00, "alias": ["colores", "caja de colores", "colores de madera"]},
    "crayones":                      {"precio": 30.00, "alias": ["crayolas", "crayones", "caja de crayolas"]},
    "plumones":                      {"precio": 52.00, "alias": ["plumones", "marcadores", "plumon"]},
}

# Mapa alias -> clave canónica (para resolver sinónimos hablados)
ALIAS = {}
for clave, datos in CATALOGO.items():
    ALIAS[clave.lower()] = clave
    for a in datos.get("alias", []):
        ALIAS[a.lower()] = clave

print(f"✔ Catálogo cargado: {len(CATALOGO)} artículos")
for k, v in CATALOGO.items():
    print(f"  · {k}: ${v['precio']:.2f}")

## 💾 Paso 3 · La base de datos de ventas (SQLite)
Tabla `ventas` (una fila por venta) y `detalle` (cada artículo de cada venta). Historial relacional listo para el dashboard.

In [ ]:
import sqlite3, datetime
import pandas as pd

DB = "ventas_papeleria.db"
conn = sqlite3.connect(DB)
c = conn.cursor()

SQL = '''CREATE TABLE IF NOT EXISTS ventas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha TEXT NOT NULL,
    total REAL NOT NULL,
    num_articulos INTEGER NOT NULL
);
CREATE TABLE IF NOT EXISTS detalle (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    venta_id INTEGER NOT NULL,
    articulo TEXT NOT NULL,
    cantidad INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    subtotal REAL NOT NULL,
    FOREIGN KEY (venta_id) REFERENCES ventas(id)
);'''
c.executescript(SQL)
conn.commit()

def guardar_venta(items, total):
    """items: lista de dicts {articulo, cantidad, precio_unitario, subtotal}."""
    fecha = datetime.datetime.now().isoformat(timespec="seconds")
    num = sum(i["cantidad"] for i in items)
    c.execute("INSERT INTO ventas (fecha, total, num_articulos) VALUES (?, ?, ?)",
              (fecha, total, num))
    venta_id = c.lastrowid
    for i in items:
        c.execute("INSERT INTO detalle (venta_id, articulo, cantidad, precio_unitario, subtotal) "
                  "VALUES (?, ?, ?, ?, ?)",
                  (venta_id, i["articulo"], i["cantidad"], i["precio_unitario"], i["subtotal"]))
    conn.commit()
    return venta_id, fecha

def leer_historial():
    return pd.read_sql_query("SELECT * FROM ventas ORDER BY id DESC", conn)

print("✔ Base de datos lista:", DB)
print("Ventas guardadas hasta ahora:", leer_historial().shape[0])

## 🎙️ Paso 4 · El parser: de lo que dijo el cliente a (artículo, cantidad)
Cuando el cliente dice *"5 cuadernos y 3 plumas azules"*, hay que separar **qué** pidió y **cuánto**. Usamos **regex** para números (en dígitos o palabras) y buscamos el artículo en el catálogo con alias/sinónimos.

In [ ]:
import re

PALABRAS_NUM = {
    "un": 1, "una": 1, "uno": 1, "dos": 2, "tres": 3, "cuatro": 4, "cinco": 5,
    "seis": 6, "siete": 7, "ocho": 8, "nueve": 9, "diez": 10, "once": 11,
    "doce": 12, "quince": 15, "veinte": 20, "treinta": 30, "cincuenta": 50, "cien": 100,
}

def _num_a_int(txt):
    txt = txt.strip().lower()
    if txt.isdigit():
        return int(txt)
    return PALABRAS_NUM.get(txt, 1)

def parsear_pedido(texto):
    """Regresa (items, no_reconocido). items: dicts con articulo/cantidad/precio_unitario/subtotal."""
    texto = texto.lower()
    items = []
    no_reconocido = []

    # separa por "y", "," o "y despues"
    segmentos = re.split(r"\s*(?:y\s+|,\s*|,|\s+y\s+)\s*", texto)
    for seg in segmentos:
        seg = seg.strip()
        if not seg:
            continue
        m = re.match(
            r"^(\d+|un|una|uno|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|diez|once|doce|quince|veinte|treinta|cincuenta|cien)\s+(.+)$",
            seg)
        if m:
            cantidad = _num_a_int(m.group(1))
            nombre = m.group(2).strip()
        else:
            cantidad = 1
            nombre = seg

        # limpiar palabras ruido
        nombre = re.sub(r"\b(de|del|por favor|porfa|favor|me das|quiero|dame|necesito|ocupo|una|un)\b", " ", nombre)
        nombre = re.sub(r"\s+", " ", nombre).strip()

        clave = ALIAS.get(nombre)
        if clave is None:
            for alias, canon in ALIAS.items():
                if alias in nombre or nombre in alias:
                    clave = canon
                    break
        if clave:
            precio = CATALOGO[clave]["precio"]
            items.append({
                "articulo": clave,
                "cantidad": cantidad,
                "precio_unitario": precio,
                "subtotal": round(precio * cantidad, 2),
            })
        else:
            no_reconocido.append(nombre)

    return items, no_reconocido

# Pruebas rápidas
for frase in ["cinco cuadernos y tres plumas azules", "2 lapices y una goma de borrar",
              "5 cuadernos rayados, 3 plumas y 2 marcatextos", "una calculadora cientifica y 20 hojas"]:
    its, faltan = parsear_pedido(frase)
    print("🎙️", repr(frase))
    for i in its:
        print(f"    ✓ {i['cantidad']} x {i['articulo']} = ${i['subtotal']:.2f}")
    if faltan:
        print("    ⚠ no reconocido:", faltan)
    print()

## 🎤 Paso 5 · Reconocimiento de voz (voz → texto)
Función que graba audio del micrófono y lo convierte a texto con **Google Speech Recognition**. En Colab, Gradio maneja el micrófono directamente, pero esta función sirve para probar la lógica por separado (útil también en local).

In [ ]:
import speech_recognition as sr

rec = sr.Recognizer()

def voz_a_texto(audio_file=None):
    """audio_file: ruta a .wav/.flac (el que sube Gradio) o None para usar el micrófono."""
    if audio_file is not None:
        with sr.AudioFile(audio_file) as fuente:
            audio = rec.record(fuente)
    else:
        with sr.Microphone() as fuente:
            print("🎙️ Habla ahora…")
            audio = rec.listen(fuente)
    try:
        return rec.recognize_google(audio, language="es-MX")
    except sr.UnknownValueError:
        return "No entendí, ¿puedes repetirlo?"
    except sr.RequestError as e:
        return f"Error de reconocimiento: {e}"

print("✔ Módulo de voz listo")

## 🔊 Paso 5b · Sintetizar voz (texto → voz): leer el total en voz alta
Además de escuchar al cliente, la papelería puede **hablar**: al cerrar la venta leerá el **total en voz alta**. Usamos `gTTS` (Google TTS, gratis) y `IPython.display.Audio` para reproducirlo en Colab y en la app.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display
import tempfile, os

def hablar(texto, lang="es"):
    """Genera audio de un texto y regresa la ruta del archivo mp3."""
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tmp.close()
    tts = gTTS(text=texto, lang=lang)
    tts.save(tmp.name)
    return tmp.name

def reproducir_en_colab(texto):
    """Reproduce el audio directamente en el notebook."""
    archivo = hablar(texto)
    display(Audio(archivo, autoplay=True))
    return archivo

def total_a_texto(total):
    """Convierte un total numérico en una frase en español mexicano legible."""
    pesos = int(total)
    centavos = round((total - pesos) * 100)
    if centavos:
        return f"{pesos} pesos con {centavos} centavos."
    return f"{pesos} pesos."

# Prueba rápida
prueba = hablar("Son doscientos treinta y cuatro pesos. ¡Gracias por tu compra!")
display(Audio(prueba, autoplay=True))
print("✔ Módulo de TTS listo. Escuchaste la prueba 👆")

## 🧾 Paso 6 · El carrito y el total de la venta
Un objeto `Venta` que acumula artículos, calcula subtotales y el **total**. Al llamar `cerrar_venta()` calcula el total final y lo guarda en SQLite, dejando el carrito listo para la siguiente venta.

In [ ]:
class Venta:
    def __init__(self):
        self.items = []

    def agregar_items(self, lista):
        """lista: salida de parsear_pedido (dicts con articulo/cantidad/precio_unitario/subtotal)."""
        self.items.extend(lista)

    def total(self):
        return round(sum(i["subtotal"] for i in self.items), 2)

    def num_articulos(self):
        return sum(i["cantidad"] for i in self.items)

    def cerrar_venta(self):
        """Calcula total final y guarda en SQLite. Regresa (venta_id, fecha, total)."""
        total = self.total()
        if not self.items:
            return None, None, total
        venta_id, fecha = guardar_venta(self.items, total)
        return venta_id, fecha, total

    def vaciar(self):
        self.items = []

    def tabla(self):
        return pd.DataFrame(self.items)

carrito = Venta()
print("✔ Carrito listo")

In [ ]:
# Demo: dictar dos pedidos, ver el carrito y cerrar la primera venta
for frase in ["dos cuadernos profesionales y cinco plumas", "una calculadora cientifica"]:
    its, faltan = parsear_pedido(frase)
    carrito.agregar_items(its)
    print("➕ Agregado:", frase)
    if faltan:
        print("   ⚠ no reconocido:", faltan)

print("\n🧾 Carrito actual:")
print(carrito.tabla().to_string(index=False))
print("\nTotal hasta ahora: $%.2f" % carrito.total())

print("\n💰 Cerrando venta…")
venta_id, fecha, total = carrito.cerrar_venta()
print(f"   Venta #{venta_id} registrada · {fecha} · Total: ${total:.2f}")

# 🔉 Lee el total en voz alta
frase_total = f"Tu total es de {total_a_texto(total)} ¡Gracias por tu compra!"
reproducir_en_colab(frase_total)

carrito.vaciar()
print("   (carrito vaciado para la siguiente venta ✓)")

## 📊 Paso 7 · Historial de ventas y dashboard
Leemos la base de datos y construimos gráficas: **ventas por día**, **artículos más vendidos** y **resumen de totales**, con pandas + plotly (interactivo).

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def cargar_datos():
    ventas = pd.read_sql_query("SELECT * FROM ventas ORDER BY id", conn)
    detalle = pd.read_sql_query("SELECT * FROM detalle", conn)
    if ventas.empty:
        return ventas, detalle
    ventas["fecha_dt"] = pd.to_datetime(ventas["fecha"])
    ventas["dia"] = ventas["fecha_dt"].dt.date
    return ventas, detalle

def dashboard():
    ventas, detalle = cargar_datos()
    if ventas.empty:
        print("📭 Aún no hay ventas. Haz una venta en el Paso 6 o en la app para ver el dashboard.")
        return

    por_dia = ventas.groupby("dia")["total"].sum().reset_index()
    fig1 = px.bar(por_dia, x="dia", y="total",
                  labels={"dia": "Día", "total": "Total vendido ($)"},
                  title="💰 Ventas por día", text_auto=True)

    if not detalle.empty:
        por_art = detalle.groupby("articulo")["cantidad"].sum().reset_index().sort_values("cantidad", ascending=False)
        fig2 = px.bar(por_art, x="cantidad", y="articulo", orientation="h",
                      labels={"cantidad": "Unidades vendidas", "articulo": "Artículo"},
                      title="🛒 Artículos más vendidos", text_auto=True)
        fig2.update_yaxes(autorange="reversed")
    else:
        fig2 = go.Figure()

    total_vendido = ventas["total"].sum()
    num_ventas = len(ventas)
    total_articulos = ventas["num_articulos"].sum()
    print(f"📈 Resumen · {num_ventas} ventas · {total_articulos} artículos · Total: ${total_vendido:.2f}")
    print(f"Ticket promedio: ${total_vendido/num_ventas:.2f}")

    fig1.show()
    fig2.show()
    return ventas, detalle

dashboard()

## 🖥️ Paso 8 · La app de Gradio (punto de venta con voz + dashboard)
La interfaz final:
- 🎙️ **Micrófono** para dictar el pedido (o caja de texto por si prefiere escribir).
- 🧾 **Carrito** en vivo con el total.
- 💰 Botón **"Cerrar venta"** que guarda todo, **lee el total en voz alta** y reinicia.
- 📊 Pestaña con el **dashboard** de ventas.

In [ ]:
import gradio as gr

ESTADO = {"carrito": Venta()}

def _lineas_items():
    if not ESTADO["carrito"].items:
        return "**Carrito vacío.** Dicta tu primer artículo. 🎙️"
    df = ESTADO["carrito"].tabla()
    df["subtotal"] = df["subtotal"].map("${:,.2f}".format)
    df["precio_unitario"] = df["precio_unitario"].map("${:,.2f}".format)
    try:
        lineas = df.to_markdown(index=False)
    except Exception:
        lineas = df.to_string(index=False)
    total = ESTADO["carrito"].total()
    return f"{lineas}\n\n**Total: ${total:,.2f}**"

def procesar_voz(frase, audio):
    texto = frase or ""
    if audio is not None:
        voz = voz_a_texto(audio)
        texto = (texto + " " + voz).strip()
    if not texto:
        return _lineas_items(), "Di algo o escribe un pedido."
    its, faltan = parsear_pedido(texto)
    ESTADO["carrito"].agregar_items(its)
    msgs = []
    for i in its:
        msgs.append(f"➕ {i['cantidad']} × {i['articulo']} = ${i['subtotal']:.2f}")
    if faltan:
        msgs.append("⚠ No reconocí: " + ", ".join(faltan))
    detalle = "\n".join(msgs) if msgs else "No se agregó nada."
    return _lineas_items(), detalle

def cerrar_venta():
    venta_id, fecha, total = ESTADO["carrito"].cerrar_venta()
    if venta_id is None:
        detalle = "No hay artículos que cobrar."
        return _lineas_items(), detalle, None
    detalle = f"💰 Venta #{venta_id} cerrada el {fecha} · **Total: ${total:,.2f}**"
    ESTADO["carrito"].vaciar()
    # 🔉 Genera y regresa el audio con el total para reproducirlo
    frase_total = f"Tu total es de {total_a_texto(total)} ¡Gracias por tu compra!"
    archivo_audio = hablar(frase_total)
    return _lineas_items(), detalle, archivo_audio

def pestana_dashboard():
    ventas, detalle = cargar_datos()
    if ventas.empty:
        return "📭 Aún no hay ventas registradas.", None, None
    total_vendido = ventas["total"].sum()
    num_ventas = len(ventas)
    por_dia = ventas.groupby("dia")["total"].sum().reset_index()
    fig1 = px.bar(por_dia, x="dia", y="total", labels={"dia": "Día", "total": "Total ($)"},
                  title="Ventas por día", text_auto=True)
    if not detalle.empty:
        por_art = detalle.groupby("articulo")["cantidad"].sum().reset_index().sort_values("cantidad", ascending=False)
        fig2 = px.bar(por_art, x="cantidad", y="articulo", orientation="h",
                      labels={"cantidad": "Unidades", "articulo": "Artículo"},
                      title="Artículos más vendidos", text_auto=True)
        fig2.update_yaxes(autorange="reversed")
    else:
        fig2 = go.Figure()
    resumen = (f"### 📈 Resumen\n{num_ventas} ventas · Total ${total_vendido:,.2f} · "
               f"Ticket promedio ${total_vendido/num_ventas:,.2f}")
    return resumen, fig1, fig2

with gr.Blocks(title="Papelería La Escuadra · Punto de venta con voz") as app:
    gr.Markdown("# ✂️ Papelería La Escuadra\n_Punto de venta con voz — dicta, cobra y guarda tus ventas._")
    with gr.Tabs():
        with gr.Tab("🧾 Venta"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 🎙️ Dicta el pedido del cliente")
                    voz = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Micrófono")
                    frase = gr.Textbox(label="…o escríbelo", placeholder="Ej: cinco cuadernos y tres plumas", lines=2)
                    gr.Examples(["cinco cuadernos y tres plumas",
                                 "dos lapices, una goma y un sacapuntas",
                                 "una calculadora cientifica y veinte hojas"], inputs=frase)
                    boton_cobrar = gr.Button("💰 Cerrar venta", variant="primary")
                with gr.Column():
                    gr.Markdown("### 🧾 Carrito")
                    carrito_md = gr.Markdown(_lineas_items())
                    detalle_md = gr.Markdown("")
                    audio_total = gr.Audio(label="🔊 Total en voz alta", type="filepath", autoplay=True)
            voz.change(procesar_voz, inputs=[frase, voz], outputs=[carrito_md, detalle_md])
            frase.submit(procesar_voz, inputs=[frase, voz], outputs=[carrito_md, detalle_md])
            boton_cobrar.click(cerrar_venta, outputs=[carrito_md, detalle_md, audio_total])
        with gr.Tab("📊 Dashboard"):
            boton_refrescar = gr.Button("🔄 Actualizar")
            resumen = gr.Markdown()
            fig1 = gr.Plot()
            fig2 = gr.Plot()
            boton_refrescar.click(pestana_dashboard, outputs=[resumen, fig1, fig2])
            app.load(pestana_dashboard, outputs=[resumen, fig1, fig2])

app.launch(share=True, debug=False)

## 🎯 Lo que puedes personalizar
1. **Agrega o quita artículos** en `CATALOGO` (con sus alias para que la voz los entienda).
2. **Mejora el parser** para frases más complejas ("dame una caja de colores y dos de plumas").
3. **Agrega IVA/descuentos en `cerrar_venta()`** (p.ej. 5% de descuento a estudiantes).
4. **Exporta el historial** a Excel con `leer_historial().to_excel("ventas.xlsx")`.
5. **Lleva el dashboard a externos** (Power BI / Looker Studio) leyendo `ventas_papeleria.db` o exportando CSV.

Acabas de construir un **punto de venta por voz con historial y dashboard**: voz → catálogo → carrito → total → SQLite → gráficas. 🖊️📊✨